In [1]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("rmisra/news-category-dataset")

df = pd.read_json(path + '/News_Category_Dataset_v3.json', lines=True)

print(df.shape)
print("Path to dataset files:", path)

100%|██████████| 26.5M/26.5M [00:01<00:00, 15.4MB/s]

Extracting files...


(209527, 6)
Path to dataset files: /home/jens/.cache/kagglehub/datasets/rmisra/news-category-dataset/versions/3


In [2]:
df.describe()

,date
count,209527
mean,2015-04-30 00:44:14.344308
min,2012-01-28 00:00:00
25%,2013-08-10 00:00:00
50%,2015-03-16 00:00:00
75%,2016-11-01 00:00:00
max,2022-09-23 00:00:00


In [3]:
df.sample(5, random_state=42)

,link,headline,category,short_description,authors,date
128310,https://www.huffingtonpost.com/entry/what-if-w...,What If We Were All Family Generation Changers?,IMPACT,"What if, in doing so, we won't just create new...","Matt Murrie, ContributorEdupreneur, Cofounder/...",2014-06-20
139983,https://www.huffingtonpost.comhttp://www.washi...,Firestorm At AOL Over Employee Benefit Cuts,BUSINESS,It should have been a glorious week for AOL ch...,,2014-02-08
42339,https://www.huffingtonpost.com/entry/time-runs...,Dakota Access Protesters Arrested As Deadline ...,POLITICS,A few protesters who refused to leave remained...,"Michael McLaughlin & Josh Morgan, The Huffingt...",2017-02-22
131494,https://www.huffingtonpost.com/entry/one-glimp...,One Glimpse Of These Baby Kit Foxes And You'll...,GREEN,,,2014-05-14
163649,https://www.huffingtonpost.com/entry/mens-swea...,"Mens' Sweat Pheromone, Androstadienone, Influe...",SCIENCE,Scientists didn't know if humans played that g...,Melissa Cronin,2013-06-02


In [4]:
print("Number of unique categories:", df['category'].nunique())
print("\nCategory counts:")
print(df['category'].value_counts())

Number of unique categories: 42

Category counts:
category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATIN

In [5]:
df['combined_text'] = df['headline'] + ' ' + df['short_description']

# Clean dataset
print(df.shape)
clean_df = df[['combined_text', 'category']].copy()
clean_df.dropna(inplace=True)
clean_df.drop_duplicates(inplace=True)
print(clean_df.shape)
clean_df.head(5)

(209527, 7)
(209056, 2)


,combined_text,category
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY
3,The Funniest Tweets From Parents This Week (Se...,PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS


In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def preprocess_text(text):
    sanitized_text = re.sub(r'[^a-zA-Z\s]', '', text)
    sanitized_text = sanitized_text.lower()
    split_text = sanitized_text.split()
    text_without_stopwords = [word for word in split_text if word not in stop_words]
    lemmatized_text = [lemmatizer.lemmatize(word) for word in text_without_stopwords]
    return ' '.join(lemmatized_text)

clean_df['combined_text'] = clean_df['combined_text'].apply(preprocess_text)
clean_df.head(5)

[nltk_data] Downloading package stopwords to /home/jens/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jens/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,combined_text,category
0,million american roll sleeve omicrontargeted c...,U.S. NEWS
1,american airline flyer charged banned life pun...,U.S. NEWS
2,funniest tweet cat dog week sept dog dont unde...,COMEDY
3,funniest tweet parent week sept accidentally p...,PARENTING
4,woman called cop black birdwatcher loses lawsu...,U.S. NEWS


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=50000)
tfidf_matrix = vectorizer.fit_transform(clean_df['combined_text'])

# TF IDF = Term Frequency Inverse Document Frequency
print("Shape of TF-IDF matrix:", tfidf_matrix.shape)

Shape of TF-IDF matrix: (209056, 50000)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(clean_df['category'])

x = tfidf_matrix
y = y_encoded

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("Classes:", label_encoder.classes_)
print("Train size:", x_train.shape[0])
print("Test size:", x_test.shape[0])

Classes: ['ARTS' 'ARTS & CULTURE' 'BLACK VOICES' 'BUSINESS' 'COLLEGE' 'COMEDY'
 'CRIME' 'CULTURE & ARTS' 'DIVORCE' 'EDUCATION' 'ENTERTAINMENT'
 'ENVIRONMENT' 'FIFTY' 'FOOD & DRINK' 'GOOD NEWS' 'GREEN' 'HEALTHY LIVING'
 'HOME & LIVING' 'IMPACT' 'LATINO VOICES' 'MEDIA' 'MONEY' 'PARENTING'
 'PARENTS' 'POLITICS' 'QUEER VOICES' 'RELIGION' 'SCIENCE' 'SPORTS' 'STYLE'
 'STYLE & BEAUTY' 'TASTE' 'TECH' 'THE WORLDPOST' 'TRAVEL' 'U.S. NEWS'
 'WEDDINGS' 'WEIRD NEWS' 'WELLNESS' 'WOMEN' 'WORLD NEWS' 'WORLDPOST']
Train size: 167244
Test size: 41812


In [1]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, solver='lbfgs')
model.fit(x_train, y_train)

NameError: name 'x_train' is not defined

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_pred = model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy:  {accuracy * 100 :.2f}%")
print(f"Precision: {precision * 100 :.2f}%")
print(f"Recall:    {recall * 100 :.2f}%")
print(f"F1-score:  {f1 * 100 :.2f}%")

print("\nClassification Report:\n",
      classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
import joblib
from sklearn.pipeline import Pipeline

joblib.dump(model, f'model_acc_{accuracy * 100 :.2f}%.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')

pipeline = Pipeline([
    ('tfidf', vectorizer),
    ('clf', model)
])

joblib.dump(pipeline, f'pipeline_acc_{accuracy * 100:.2f}%.pkl')